In [1]:
pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 12.4 MB/s eta 0:00:00


In [2]:
import random
import numpy as np
import torch

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Use the function to set the seed
set_seed(42)


In [3]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import KarateClub
from torch_geometric.nn import GCNConv, global_mean_pool

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

dataset = KarateClub()
data = dataset[0]

num_train_nodes = int(0.8 * data.num_nodes)
data.train_mask = torch.zeros(data.num_nodes, dtype=bool)
data.train_mask[:num_train_nodes] = True
data.test_mask = ~data.train_mask

class GNN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(GNN, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)
        self.bn1 = nn.BatchNorm1d(hidden_channels)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.out_channels = out_channels

    def forward(self, x, edge_index):
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = self.bn2(self.conv2(x, edge_index))
        return x

class GNN_KNN_MLP(nn.Module):
    def __init__(self, gnn, mlp_hidden_dim, num_classes, k=2):
        super(GNN_KNN_MLP, self).__init__()
        self.gnn = gnn
        self.k = k
        self.pool = global_mean_pool
        self.mlp = nn.Sequential(
            nn.Linear(gnn.out_channels, mlp_hidden_dim),
            nn.ReLU(),
            nn.Linear(mlp_hidden_dim, mlp_hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(mlp_hidden_dim, 1),
        )
        self.classifier = nn.Linear(gnn.out_channels, num_classes)

    def forward(self, x, edge_index, batch):
        embeddings = self.gnn(x, edge_index)
        distances = torch.cdist(embeddings, embeddings)
        knn_indices = torch.topk(-distances, self.k , dim=-1)[1][:, 1:]

        out_logits = torch.zeros((embeddings.size(0), self.classifier.out_features), device=x.device)

        complex_sets = []  # Store sets selected by MLP

        for i in range(embeddings.size(0)):
            knn_set = torch.cat((embeddings[knn_indices[i]], embeddings[i].unsqueeze(0)), dim=0)
            pooled_embedding = self.pool(knn_set, batch=None)

            include_prob = torch.sigmoid(self.mlp(pooled_embedding)).view(-1)
            inclusion_sample = (torch.rand_like(include_prob) < include_prob).float()
            straight_through_sample = inclusion_sample + (include_prob - include_prob.detach())

            selected_embedding = straight_through_sample * pooled_embedding + (1 - straight_through_sample) * embeddings[i]

            # Identify nodes in the current knn_set selected by MLP
            if inclusion_sample.item() == 1.0:
                complex_sets.append(tuple(knn_indices[i].tolist() + [i]))  # Save as tuple of node IDs

            out_logits[i] = self.classifier(selected_embedding)

        print(f"High-order sets selected by MLP: {complex_sets}")  # Shows node sets selected

        return F.log_softmax(out_logits, dim=-1)

num_classes = data.y.max().item() + 1
gnn = GNN(in_channels=dataset.num_node_features, hidden_channels=16, out_channels=8)
model = GNN_KNN_MLP(gnn, mlp_hidden_dim=16, num_classes=num_classes, k=3)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
data = data.to(device)

def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index, data.batch)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Gradient clipping
    optimizer.step()

    # Logging gradients
    # for name, param in model.named_parameters():
    #     if param.grad is not None:
    #         print(f"{name} gradient: {param.grad}")

    #print(f"Training Loss: {loss.item()}")
    return loss.item()

def test():
    model.eval()
    correct = 0
    out = model(data.x, data.edge_index, data.batch)
    pred = out.argmax(dim=1)
    correct = (pred[data.test_mask] == data.y[data.test_mask]).sum().item()
    test_size = data.test_mask.sum().item()
    accuracy = correct / test_size if test_size > 0 else 0.0
    return accuracy

for epoch in range(1, 1000):
    train_loss = train()
    test_acc = test()
    print(f"Epoch {epoch}: Train Loss = {train_loss:.4f}, Test Accuracy = {test_acc:.4f}\n")


High-order sets selected by MLP: [(13, 3, 2), (7, 12, 3), (10, 5, 4), (6, 10, 5), (16, 5, 6), (18, 30, 8), (30, 8, 9), (4, 5, 10), (19, 25, 11), (22, 23, 14), (8, 30, 15), (6, 5, 16), (8, 27, 18), (18, 8, 19), (23, 14, 20), (14, 23, 22), (25, 24, 23), (29, 23, 26), (24, 27, 28), (26, 23, 29), (24, 23, 31)]
High-order sets selected by MLP: [(30, 13, 2), (6, 16, 5), (16, 5, 6), (30, 8, 9), (19, 6, 11), (3, 4, 12), (22, 15, 14), (14, 18, 15), (6, 5, 16), (13, 7, 17), (22, 23, 20), (28, 24, 23), (25, 23, 24), (24, 23, 25), (28, 23, 27), (23, 27, 28), (26, 20, 29), (9, 8, 30), (32, 8, 33)]
Epoch 1: Train Loss = 1.5540, Test Accuracy = 0.0000

High-order sets selected by MLP: [(19, 7, 13), (22, 23, 14), (8, 30, 18), (14, 23, 22), (24, 23, 25), (29, 23, 26), (24, 27, 28), (8, 18, 30), (31, 27, 32)]
High-order sets selected by MLP: [(1, 19, 0), (7, 13, 3), (30, 8, 9), (3, 4, 12), (14, 18, 15), (7, 13, 17), (23, 25, 19), (23, 22, 20), (14, 20, 22), (25, 23, 24), (24, 23, 25), (29, 20, 26), (27,

KeyboardInterrupt: 

#Including TNN


In [2]:
pip install toponetx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 703.7/703.7 kB 18.7 MB/s eta 0:00:00


In [3]:
pip install topomodelx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.9/62.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.9/107.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 39.1 MB/s eta 0:00:00


In [8]:
import torch_geometric

ModuleNotFoundError: No module named 'torch'

In [20]:
def generate_faces(simplex, max_dim):
    faces = {0: [[v] for v in simplex]}  # Começa com vértices (0-faces)
    for dim in range(1, max_dim + 1):
        faces[dim] = []
        # Construindo faces de `dim` com base nas faces de `dim-1`
        for face in faces[dim - 1]:
            for v in simplex:
                if v > max(face):  # Adiciona vértices maiores para evitar duplicação
                    new_face = face + [v]
                    faces[dim].append(new_face)

    for dim in faces:
        faces[dim] = torch.tensor(faces[dim], dtype=torch.long)

    return faces


In [21]:
def remove_duplicated_edges(edge_index):
    arestas = set()
    for i in range(edge_index.size(1)):
        u = edge_index[0, i].item()
        v = edge_index[1, i].item()
        arestas.add((min(u, v), max(u, v)))  # Armazenar como um par ordenado
    return arestas

78

In [26]:
arestas_tensor = torch.tensor(list(remove_duplicated_edges(data.edge_index)), dtype=torch.long)

In [55]:
def get_faces_triangles(triangulos):
    faces_triangulos = {}
    for triangle in triangulos:
        faces_triangulos[triangle] = generate_faces(triangle, 2)[1]
    return faces_triangulos


In [68]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import KarateClub
from torch_geometric.nn import GCNConv, global_mean_pool
import toponetx as tnx
from topomodelx.nn.simplicial.san import SAN
from topomodelx.utils.sparse import from_sparse

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

dataset = KarateClub()
data = dataset[0]

num_train_nodes = int(0.8 * data.num_nodes)
data.train_mask = torch.zeros(data.num_nodes, dtype=bool)
data.train_mask[:num_train_nodes] = True
data.test_mask = ~data.train_mask

class GNN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(GNN, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)
        self.bn1 = nn.BatchNorm1d(hidden_channels)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.out_channels = out_channels

    def forward(self, x, edge_index):
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = self.bn2(self.conv2(x, edge_index))
        return x

class TNN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.base_model = SAN(in_channels, hidden_channels, n_layers=2)
        self.linear = nn.Linear(hidden_channels, out_channels)

    def forward(self, x, laplacian_up, laplacian_down):
        x = self.base_model(x, laplacian_up, laplacian_down)
        return torch.sigmoid(self.linear(x))

class GNN_KNN_MLP(nn.Module):
    def __init__(self, gnn, mlp_hidden_dim, tnn_hidden_dim, num_classes, k=2, max_dim=2):
        super(GNN_KNN_MLP, self).__init__()
        self.gnn = gnn
        self.k = k
        self.pool = global_mean_pool
        self.mlp = nn.Sequential(
            nn.Linear(gnn.out_channels, mlp_hidden_dim),
            nn.ReLU(),
            nn.Linear(mlp_hidden_dim, mlp_hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(mlp_hidden_dim, 1),
        )
        self.classifier = nn.Linear(gnn.out_channels, num_classes)
        self.tnn = TNN(in_channels=gnn.out_channels, hidden_channels=tnn_hidden_dim, out_channels=num_classes)
        self.max_dim = max_dim

    def forward(self, x, edge_index, batch):
        embeddings = self.gnn(x, edge_index)
        distances = torch.cdist(embeddings, embeddings)
        knn_indices = torch.topk(-distances, self.k, dim=-1)[1]
        out_logits = torch.zeros((embeddings.size(0), self.classifier.out_features), device=x.device)
        complex_sets = []

        self.__init_boundary_matrices(edge_index, knn_indices)

        for i in range(embeddings.size(0)):
            knn_set = torch.cat((embeddings[knn_indices[i]], embeddings[i].unsqueeze(0)), dim=0)
            pooled_embedding = self.pool(knn_set, batch=None)

            include_prob = torch.sigmoid(self.mlp(pooled_embedding)).view(-1)
            inclusion_sample = (torch.rand_like(include_prob) < include_prob).float()
            straight_through_sample = inclusion_sample + (include_prob - include_prob.detach())


            for triangulo, face in simplexes.items():
                for i in range(edge_index_with_unique_edges.size(1)):
                    if edge_index_with_unique_edges[:, i] in face:
                        getattr(self, f"boundary_matrizes_2")[i, simplexes.index(triangulo)] = straight_through_sample
                    matriz_arestas_triângulos[i, triangulos.index(triangulo)] = 1
            # selected_embedding = straight_through_sample * pooled_embedding + (1 - straight_through_sample) * embeddings[i]
           
            if inclusion_sample.item() == 1.0:
                complex_sets.append(tuple(knn_indices[i].tolist()))

            # out_logits[i] = self.classifier(selected_embedding)


        # print(edge_index.shape)
        # print(edge_index.shape)
        # print(embeddings.shape)
        # print(knn_set.shape)
        # sc = tnx.SimplicialComplex()
        # for edge in edge_index.T.tolist():
        #     sc.add_simplex(edge, rank=1)
        # for higher_order_set in complex_sets:
        #     # print(higher_order_set)
        #     sc.add_simplex(higher_order_set, rank=2)

        # B1 = sc.incidence_matrix(rank=1)
        # B2 = sc.incidence_matrix(rank=2)
        # L1_up = from_sparse(B1.T @ B1)
        # L1_down = from_sparse(B2 @ B2.T)

        # tnn_output = self.tnn(embeddings, laplacian_up=L1_up, laplacian_down=L1_down)

        return True #F.log_softmax(tnn_output, dim=-1)

    def __init_boundary_matrices(self, edge_index, simplexes):
        edge_index_with_unique_edges = remove_duplicated_edges(edge_index)
        edge_set_len = len(edge_index_with_unique_edges)
        # # print(simplexes)
        for dim in range(1, self.max_dim+1):
            if getattr(self, f'boundary_matrizes_{dim}', None) is None:
                if not dim > 1:
                   setattr(self, f'boundary_matrizes_{dim}', torch.zeros((edge_index[:,0].unique(return_counts=True), edge_set_len), device=edge_index.device, requires_grad=True))
                else: 
                   setattr(self, f'boundary_matrizes_{dim}', torch.zeros((edge_set_len, 0), device=edge_index.device, requires_grad=True))


        print(getattr(self, f'boundary_matrizes_{dim}', None).shape)
num_classes = data.y.max().item() + 1
gnn = GNN(in_channels=dataset.num_node_features, hidden_channels=16, out_channels=8)
model = GNN_KNN_MLP(gnn, mlp_hidden_dim=16, tnn_hidden_dim=16, num_classes=num_classes, k=3)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

#device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#model = model.to(device)
#data = data.to(device)

def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index, data.batch)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Gradient clipping
    optimizer.step()

    #Logging gradients
    for name, param in model.named_parameters():
        if param.grad is not None:
            print(f"{name} gradient: {param.grad}")

    print(f"Training Loss: {loss.item()}")
    return loss.item()

def test():
    model.eval()
    correct = 0
    out = model(data.x, data.edge_index, data.batch)
    pred = out.argmax(dim=1)
    correct = (pred[data.test_mask] == data.y[data.test_mask]).sum().item()
    test_size = data.test_mask.sum().item()
    accuracy = correct / test_size if test_size > 0 else 0.0
    return accuracy

for epoch in range(1, 1000):
    train_loss = train()
    test_acc = test()
    print(f"Epoch {epoch}: Train Loss = {train_loss:.4f}, Test Accuracy = {test_acc:.4f}\n")


torch.Size([34, 3])


NameError: name 'x' is not defined

In [67]:
model.boundary_matrizes_2.shape

torch.Size([78, 13])